
# Stage 10 structure-conditioned redesign on SageMaker

This notebook upgrades PhageForge from proxy-constrained sequence search to **true structure-conditioned redesign**.

The Stage 10 flow is:
1. build a structure-conditioned redesign context around the selected seed scaffold,
2. run inverse-folding beam search on the fixed backbone,
3. prefilter the strongest diverse candidates,
4. run the corrected heavy structural validator on top-10 and top-3,
5. build the final Stage 10 report.


In [ ]:

from pathlib import Path
import os, subprocess, sys

def find_repo_root(start: Path) -> Path:
    """Find the repo root even when the notebook is opened outside the unpacked repo folder."""
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'scripts').exists():
            return candidate
    for candidate in sorted(start.iterdir()):
        if candidate.is_dir() and (candidate / 'pyproject.toml').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repo root containing pyproject.toml and scripts/.')

ROOT = find_repo_root(Path.cwd())
os.chdir(ROOT)
print('Repo root:', ROOT)


In [ ]:

subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'fair-esm', 'biopython'], check=True)
print('Environment ready.')


In [ ]:

from pathlib import Path

# ----------------------------- Edit these paths once before running Stage 10 ----------------------------- #
STAGE07_CONTEXT = Path('results/stage07/context/stage07_context.base.json')
STRICT_CSV = Path('data/processed/rbp_dataset_eskapee_strict.csv')
SEED_VALIDATION_DIR = Path('results/stage09/validation_top3')

PREDICTOR_MODEL = Path('results/broad/linear_probe/seed_42/model.joblib')
LABEL_CLASSES_JSON = Path('results/broad/linear_probe/seed_42/label_classes.json')

STAGE10_DIR = Path('results/stage10')
EDIT_DIR = STAGE10_DIR / 'edit_space'
SEARCH_DIR = STAGE10_DIR / 'search'
PREFILTER_DIR = STAGE10_DIR / 'prefilter'
VAL10_DIR = STAGE10_DIR / 'validation_top10'
VAL3_DIR = STAGE10_DIR / 'validation_top3'
REPORT_DIR = STAGE10_DIR / 'final_report'

for d in [EDIT_DIR, SEARCH_DIR, PREFILTER_DIR, VAL10_DIR, VAL3_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)


In [ ]:

subprocess.run([
    sys.executable, 'scripts/10a_prepare_stage10_structure_context.py',
    '--context_json', str(STAGE07_CONTEXT),
    '--strict_csv', str(STRICT_CSV),
    '--validation_dir', str(SEED_VALIDATION_DIR),
    '--output_json', str(EDIT_DIR / 'stage10_context.json'),
    '--max_edit_positions', '6',
    '--soft_positions', '3',
    '--min_mutations', '1',
    '--max_mutations', '4',
], check=True)


In [ ]:

subprocess.run([
    sys.executable, 'scripts/10b_run_inverse_folding_beam_search.py',
    '--stage10_context_json', str(EDIT_DIR / 'stage10_context.json'),
    '--predictor_model', str(PREDICTOR_MODEL),
    '--label_classes_json', str(LABEL_CLASSES_JSON),
    '--embedding_model', 'facebook/esm2_t33_650M_UR50D',
    '--beam_width', '24',
    '--rounds', '4',
    '--proposals_per_parent', '8',
    '--substitutions_per_position', '3',
    '--if_chain_id', 'A',
    '--if_device', 'cuda',
    '--batch_size', '4',
    '--out_csv', str(SEARCH_DIR / 'stage10_search.csv'),
    '--out_json', str(SEARCH_DIR / 'stage10_search_summary.json'),
], check=True)


In [ ]:

subprocess.run([
    sys.executable, 'scripts/10c_prefilter_stage10_candidates.py',
    '--stage10_context_json', str(EDIT_DIR / 'stage10_context.json'),
    '--search_csv', str(SEARCH_DIR / 'stage10_search.csv'),
    '--embedding_model', 'facebook/esm2_t33_650M_UR50D',
    '--batch_size', '4',
    '--top_k', '10',
    '--top_k_final', '3',
    '--out_topk_csv', str(PREFILTER_DIR / 'stage10_top10.csv'),
    '--out_topk_final_csv', str(PREFILTER_DIR / 'stage10_top3.csv'),
    '--out_json', str(PREFILTER_DIR / 'stage10_prefilter_summary.json'),
], check=True)


In [ ]:

subprocess.run([
    sys.executable, 'scripts/10d_validate_stage10_candidates.py',
    '--validated_csv', str(PREFILTER_DIR / 'stage10_top10.csv'),
    '--ranked_csv', str(SEARCH_DIR / 'stage10_search.csv'),
    '--context_json', str(EDIT_DIR / 'stage10_context.json'),
    '--out_dir', str(VAL10_DIR),
    '--top_k', '10',
    '--device', 'cuda',
    '--chunk_size', '128',
    '--num_recycles', '1',
    '--resume',
    '--out_json', str(VAL10_DIR / 'stage10_validation_top10_launch.json'),
], check=True)


In [ ]:

subprocess.run([
    sys.executable, 'scripts/10d_validate_stage10_candidates.py',
    '--validated_csv', str(PREFILTER_DIR / 'stage10_top3.csv'),
    '--ranked_csv', str(SEARCH_DIR / 'stage10_search.csv'),
    '--context_json', str(EDIT_DIR / 'stage10_context.json'),
    '--out_dir', str(VAL3_DIR),
    '--top_k', '3',
    '--device', 'cuda',
    '--chunk_size', '128',
    '--num_recycles', '1',
    '--resume',
    '--out_json', str(VAL3_DIR / 'stage10_validation_top3_launch.json'),
], check=True)


In [ ]:

VALIDATION_CSV = VAL3_DIR / 'stage08_structural_fasttrack_summary.csv'
BASELINE_VALIDATION_CSV = Path('results/stage09/validation_top3/stage08_structural_fasttrack_summary.csv')

subprocess.run([
    sys.executable, 'scripts/10e_make_stage10_report.py',
    '--stage10_context_json', str(EDIT_DIR / 'stage10_context.json'),
    '--search_csv', str(SEARCH_DIR / 'stage10_search.csv'),
    '--prefilter_csv', str(PREFILTER_DIR / 'stage10_top10.csv'),
    '--validation_csv', str(VALIDATION_CSV),
    '--baseline_validation_csv', str(BASELINE_VALIDATION_CSV),
    '--out_dir', str(REPORT_DIR),
], check=True)


In [ ]:

print('Top report files:')
for path in sorted(REPORT_DIR.glob('*')):
    print('-', path)
